## Chapter 5: Linear Regression

In [9]:
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.stats import linregress

**Perform a simple linear regression to find the m and b values that minimizes the loss (sum of squares).**

In [10]:
df = list(pd.read_csv("https://bit.ly/3C8JzrM", delimiter=",").itertuples())

n = len(df)

m = (n*sum(p.x*p.y for p in df) - sum(p.x for p in df) * sum(p.y for p in df)) / (n*sum(p.x**2 for p in df) - sum(p.x for p in df)**2)

b = (sum(p.y for p in df) /n) - m * sum(p.x for p in df) / n

print(m, b)

1.7591931481052476 4.69359654825405


**Calculate the correlation coefficient and statistical significane of this data (at 95% confidence). Is the correlation useful?**

In [11]:
# %%
df = pd.read_csv("https://bit.ly/3C8JzrM", delimiter=",")

x = df["x"].values
y = df["y"].values

result = linregress(x, y)

print("r:", result.rvalue)
print("p-value:", result.pvalue)

if result.pvalue < .05:
    print("The correlation is statistically significant at 95% confidence.")
else:
    print("The correlation is not statistically significant at 95% confidence.")

r: 0.924210062867716
p-value: 2.4452525699306082e-42
The correlation is statistically significant at 95% confidence.


**If i predict where x = 50, what is the 95% perdiction interval for the predicted value of y?**

In [12]:
x0 = 50

x = df["x"].values
y = df["y"].values

n = len(df)

result = linregress(x, y)

m = result.slope
b = result.intercept

y_pred = m * x0 + b

y_hat = m * x + b

sse = sum((y - y_hat) ** 2)
standard_error = np.sqrt(sse / (n - 2))

x_mean = np.mean(x)
x_sum_squares = sum((x - x_mean) ** 2)

t_value = t(n - 2).ppf(.975)

prediction_error = standard_error * np.sqrt(1 + (1 / n) + ((x0 - x_mean) ** 2 / x_sum_squares))

lower = y_pred - t_value * prediction_error
upper = y_pred + t_value * prediction_error

print("predicted y:", y_pred)
print("95% prediction interval:", lower, upper)

predicted y: 92.65325395351645
95% prediction interval: 50.79208640275637 134.51442150427653


**Start your regression over and do a train/test split. Feel free to experiment with cross-validation and random-fold validation. Does the linear regression perform well and consistently on the testing data? Why or why not?**

In [13]:
# %%
x = df["x"].values
y = df["y"].values

# train/test split
np.random.seed(42)

indices = np.random.permutation(len(df))

test_size = int(len(df) * .2)

test_indices = indices[:test_size]
train_indices = indices[test_size:]

x_train = x[train_indices]
y_train = y[train_indices]

x_test = x[test_indices]
y_test = y[test_indices]

result = linregress(x_train, y_train)

m = result.slope
b = result.intercept

y_train_pred = m * x_train + b
y_test_pred = m * x_test + b

def r2_score(y_true, y_pred):
    ss_res = sum((y_true - y_pred) ** 2)
    ss_tot = sum((y_true - np.mean(y_true)) ** 2)

    return 1 - (ss_res / ss_tot)

def mae(y_true, y_pred):
    return np.mean(abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

print("m:", m)
print("b:", b)

print("train r2:", r2_score(y_train, y_train_pred))
print("test r2:", r2_score(y_test, y_test_pred))

print("test mae:", mae(y_test, y_test_pred))
print("test rmse:", rmse(y_test, y_test_pred))

m: 1.7617903835083637
b: 5.754870930848199
train r2: 0.834899847703255
test r2: 0.9216835727896467
test mae: 11.472121536533741
test rmse: 15.404992689423104


In [14]:
runs = 100

r2_scores = []
rmse_scores = []

np.random.seed(42)

for i in range(runs):
    indices = np.random.permutation(len(df))

    test_size = int(len(df) * .2)

    test_indices = indices[:test_size]
    train_indices = indices[test_size:]

    x_train = x[train_indices]
    y_train = y[train_indices]
    x_test = x[test_indices]
    y_test = y[test_indices]

    result = linregress(x_train, y_train)
    m = result.slope
    b = result.intercept
    y_test_pred = m * x_test + b
    r2_scores.append(r2_score(y_test, y_test_pred))
    rmse_scores.append(rmse(y_test, y_test_pred))

print("mean r2:", np.mean(r2_scores))
print("r2 std:", np.std(r2_scores))
print("lowest r2:", min(r2_scores))
print("highest r2:", max(r2_scores), "\n")

print("mean rmse:", np.mean(rmse_scores))
print("rmse std:", np.std(rmse_scores))
print("lowest rmse:", min(rmse_scores))
print("highest rmse:", max(rmse_scores))

mean r2: 0.8446924850800203
r2 std: 0.05652256958748112
lowest r2: 0.6808964158371563
highest r2: 0.9424115633383625 

mean rmse: 20.54426492075879
rmse std: 3.6546926655844265
lowest rmse: 11.938553784664949
highest rmse: 32.9398038374545


The linear regression performs well on the testing data.

The original regression line has:
- m = 1.76
- b = 4.69

The Pearson correlation is 0.924, which means there is a strong positive linear relationship between x and y. Because this is very close to 1, the correlation is useful.

After doing one train/test split, the model gets:
- train R2 = 0.835
- test R2 = 0.922
- test MAE = 11.47
- test RMSE = 15.40

The test R2 is higher than the train R2, which can happen when the test set happens to contain points that fit the linear trend better than the training set.

To check consistency, I also used 100 random train/test splits. The average R2 is 0.845 with a standard deviation of 0.057. The lowest R2 is 0.681 and the highest R2 is 0.942.

The average RMSE is 20.54 with a standard deviation of 3.65. The lowest RMSE is 11.94 and the highest RMSE is 32.94.

Overall, the model performs well because it usually explains a large part of the variance in the data. However, the performance is not perfectly consistent across every split. This is because the dataset has noise, and some train/test splits are easier to predict than others.

So yes, the linear regression is useful for estimating the general trend, but individual predictions can still be inaccurate.